<a href="https://colab.research.google.com/github/ryankrismer/EM_field_solver/blob/main/EM_field_solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electromagnetic field solver

In [1]:
import numpy as np
import tqdm

## General functions

In [2]:
def find_dist(r, r_ref):
  """
  Function to calculate the spatial distance between 2 coordinates

  Parameters:
    r: [x, y, z] (1D numpy array)
    r_ref: [x_ref, y_ref, z_ref] (1D numpy array)

  Returns: Distance between coordinates (float)
  """
  return np.sqrt((r[0] - r_ref[0]) ** 2 + (r[1] - r_ref[1]) ** 2 + (r[2] - r_ref[2]) ** 2)

## Finding potentials from sources

In [3]:
# Defining constants and parameters

# E&M constants
c = 299792458 # Speed of light in vacuum (m * s^-1)
epsilon_0 = 8.8541878188 * 10 ** -12  # Vacuum electric permissivity (F * m^-1)

# General lattice parameters
Delta_t = 1.0 # Difference between consecutive time coordinate values (s)
N_plot = 40 ** 4  # Number of observation points to be plotted
N_t = int(N_plot ** (1 / 4))  # Size of time coordinate axis

# Time lattice points
T = np.linspace(0.0, Delta_t * (N_t - 1), N_t)  # Time axis (s)

# Source lattice parameters
Delta_L = 1.0 # Distance between adjacent source spatial coordinate values (m)
n_src = 101 # Size of each source spatial coordinate axis (should be odd to ensure lattice centering)

# Observation lattice parameters
n_rad = 10  # Number of source radii from source sphere's surface to observe out to
N_obs = 2 * int(np.round(((3 * N_plot ** (3 / 4) / (4 * np.pi)) / (1 - 1 / (1 + n_rad) ** 3)) ** (1 / 3))) + 1  # Size of each observation spatial coordinate axis
# Note: N_obs should be odd so center of lattice is origin

In [17]:
def find_phi(Q):
  """
  Function to find scalar potential

  Parameter:
    Q: charge distribution (collection of point charges) (4D numpy array)

  Returns: scalar potential (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  # Reading source lattice parameters
  n_t_src = np.size(Q, axis = 0)
  n_x_src = np.size(Q, axis = 1)
  n_y_src = np.size(Q, axis = 2)
  n_z_src = np.size(Q, axis = 3)

  # Ensuring size of time axis is same as expected
  if n_t_src != N_t:
    raise ValueError(f"Source time axis does not match expected value of {N_t}")

  # Ensuring source is sphere
  if n_x_src != n_y_src or n_x_src != n_z_src or n_y_src != n_z_src:
    raise ValueError("Source spatial axis sizes are different. They must be equal for spherical source.")

  n_src = n_x_src

  # Ensuring n_src and N_obs is odd so lattice center is origin
  if n_src % 2 != 1:
    raise ValueError("n_src is not odd, but it must be odd for lattice centering")

  if N_obs % 2 != 1:
    raise ValueError("N_obs is not odd, but it must be odd for lattice centering")

  # Setting source lattice parameters
  r_src = (n_src - 1) / 2 * Delta_L  # Radius of spherical source (m)

  # Source spatial lattice points
  X_src = np.linspace(-r_src, r_src, n_src) # Source x axis (m)
  Y_src = X_src # Source y axis (m)
  Z_src = X_src # Source z axis (m)

  # Setting observation lattice parameters
  R_obs = r_src * (1 + n_rad) # Radius of observation sphere (m)

  # Observation spatial lattice points
  X_obs = np.linspace(-R_obs, R_obs, N_obs) # Source x axis (m)
  Y_obs = X_obs # Source y axis (m)
  Z_obs = X_obs # Source z axis (m)

  lambda_sum = np.full((N_t, N_obs, N_obs, N_obs), np.nan)  # Creating 4D array w/ set number of values per observation coordinate

  # Calculating and setting values for each plottable observation point
  for t_obs in tqdm.tqdm(range(N_t)):
    for x_obs in tqdm.tqdm(range(N_obs)):
      for y_obs in range(N_obs):
        for z_obs in range(N_obs):
          x_i_obs = X_obs[x_obs]
          y_i_obs = Y_obs[y_obs]
          z_i_obs = Z_obs[z_obs]
          dist_origin = find_dist([x_i_obs, y_i_obs, z_i_obs], [0.0, 0.0, 0.0])

          if dist_origin > r_src and dist_origin <= R_obs:  # Ensuring we only save values we're going to plot
            lambda_i = []

            # Iterating through all source points
            for t_src in range(N_t):
              for x_src in range(n_x_src):
                for y_src in range(n_y_src):
                  for z_src in range(n_z_src):
                    if np.isnan(Q[t_src, x_src, y_src, z_src]): # Ensuring we don't check empty points
                      break

                    t_i_obs = T[t_obs]
                    t_i_src = T[t_src]
                    x_i_src = X_src[x_src]
                    y_i_src = Y_src[y_src]
                    z_i_src = Z_src[z_src]
                    dist_src = find_dist([x_i_obs, y_i_obs, z_i_obs], [x_i_src, y_i_src, z_i_src])
                    t_causal = dist_src / c + t_i_src

                    # Only contributions are source points causally related to observation points
                    if t_i_obs > t_causal - Delta_L / c and t_i_obs <= t_causal:
                      lambda_i.append(Q[t_src, x_src, y_src, z_src] / dist_src)
            lambda_sum[t_obs, x_obs, y_obs, z_obs] = np.sum(lambda_i) # Adding up all the source contributions

  return Delta_t / (4 * np.pi * epsilon_0) * lambda_sum

In [ ]:
# Test of scalar potential function
n_src = 11
Q = np.random.rand(N_t, n_src, n_src, n_src)
np.save("Q_test.npy", Q)

# Setting source lattice parameters
r_src = (n_src - 1) / 2 * Delta_L  # Radius of spherical source (m)

# Source spatial lattice points
X_src = np.linspace(-r_src, r_src, n_src) # Source x axis (m)
Y_src = X_src # Source y axis (m)
Z_src = X_src # Source z axis (m)

# Masking points outside sphere
for t in tqdm.tqdm(range(N_t)):
  for x in range(n_src):
    for y in range(n_src):
      for z in range(n_src):
        x_i = X_src[x]
        y_i = Y_src[y]
        z_i = Z_src[z]
        dist = find_dist([x_i, y_i, z_i], [0.0, 0.0, 0.0])

        if dist > r_src:
          Q[t, x, y, z] = np.nan
phi = find_phi(Q)
np.save("phi_test.npy", phi)
print(phi)

 96%|█████████▌| 49/51 [07:33<00:08,  4.08s/it]


## Finding fields from potentials

In [ ]:
def find_E_i(phi, A, i):
  """
  Function to find E field component from potentials phi and A_i

  Parameters:
    phi: scalar potential (4D numpy array)
    A: [A_x, A_y, A_z] components of vector potential (each is 4D numpy array)
    i: field component desired

  Returns: i component of E field (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  if i != 1 and i != 2 and i != 3:
    raise ValueError("i value not allowed. i must be a spatial index within [1, 2, 3]")

  # Finding A components based on index values
  A_i = A[i - 1]

  return -np.gradient(phi, axis = i) - np.gradient(A_i, axis = 0)

In [ ]:
def find_B_i(A, i):
  """
  Function to find B field component from potential A

  Parameters:
    A: [A_x, A_y, A_z] components of vector potential (each is 4D numpy array)
    i: field component desired

  Returns: i component of B field (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  # Finding index values to ensure cyclicity
  if i == 1:
    j = 2
    k = 3
  elif i == 2:
    j = 3
    k = 1
  elif i == 3:
    j = 1
    k = 2
  else:
    raise ValueError("i value not allowed. i must be a spatial index within [1, 2, 3]")

  # Finding A components based on index values
  A_j = A[j - 1]
  A_k = A[k - 1]

  return np.gradient(A_k, axis = j) - np.gradient(A_j, axis = k)

In [ ]:
# Test of find_E_i() w/ test phi
A_x = np.random.rand(N_t, N_obs, N_obs, N_obs)
np.save("A_x_test.npy", A_x)
A_y = np.random.rand(N_t, N_obs, N_obs, N_obs)
np.save("A_y_test.npy", A_y)
A_z = np.random.rand(N_t, N_obs, N_obs, N_obs)
np.save("A_z_test.npy", A_z)
E_x = find_E_i(phi, [A_x, A_y, A_z], 1)
np.save("E_x_test.npy", E_x)
print(f"phi is {phi}")
print(f"A_x is {A_x}")
print(f"E_x is {E_x}")

In [ ]:
# # Test of find_E_i()
# phi = np.random.rand(3, 3, 3, 3)
# A_x = np.random.rand(3, 3, 3, 3)
# A_y = np.random.rand(3, 3, 3, 3)
# A_z = np.random.rand(3, 3, 3, 3)
# E_x = find_E_i(phi, [A_x, A_y, A_z], 1)
# print(f"phi is {phi}")
# print(f"A_x is {A_x}")
# print(f"E_x is {E_x}")

phi is [[[[2.66956491e-01 7.43823742e-01 7.56221714e-02]
   [2.50109845e-01 8.36301113e-01 8.53895891e-01]
   [8.96860497e-01 5.97785260e-01 6.15562697e-01]]

  [[1.69860055e-01 1.91228852e-01 2.59489747e-01]
   [5.10966134e-01 5.37431591e-01 5.47236882e-01]
   [3.89707499e-01 2.01023836e-01 9.90021475e-01]]

  [[5.57791738e-01 9.35949601e-01 4.97128138e-01]
   [9.46682330e-04 2.29006676e-01 8.83718111e-01]
   [6.44865820e-01 9.08195211e-01 2.00311081e-01]]]


 [[[2.43993329e-01 8.87233886e-01 5.62334319e-02]
   [4.06131424e-01 6.14627965e-01 8.79666081e-01]
   [6.97659997e-01 9.20785448e-01 6.40945741e-01]]

  [[1.83696585e-01 9.92365966e-01 2.37147935e-01]
   [6.25286924e-01 3.72268980e-01 3.65590210e-02]
   [3.71309243e-01 3.22895205e-01 7.45921396e-02]]

  [[9.32511554e-01 9.18634239e-03 6.30394068e-01]
   [9.59058598e-01 5.30027320e-01 9.18742624e-01]
   [2.76324513e-01 2.52778650e-01 5.81490001e-01]]]


 [[[4.47386039e-01 6.75322616e-01 7.04572303e-01]
   [3.25059400e-01 5.236519

In [ ]:
# # Test of find_B_i()
# A_x = np.random.rand(3, 3, 3, 3)
# A_y = np.random.rand(3, 3, 3, 3)
# A_z = np.random.rand(3, 3, 3, 3)
# B_x = find_B_i([A_x, A_y, A_z], 1)
# print(f"A_x is {A_x}")
# print(f"A_y is {A_y}")
# print(f"A_z is {A_z}")
# print(f"B_x is {B_x}")

A_x is [[[[0.95307154 0.3869769  0.81464471]
   [0.92625786 0.48318043 0.09935442]
   [0.65977942 0.90902888 0.58763308]]

  [[0.78057705 0.02643401 0.23195566]
   [0.0043902  0.27951875 0.26583555]
   [0.57816752 0.5581665  0.43904707]]

  [[0.04150032 0.05817008 0.25300806]
   [0.72436458 0.44803979 0.56860463]
   [0.97810475 0.51435    0.91890006]]]


 [[[0.80400318 0.20275559 0.40467763]
   [0.17694077 0.95141178 0.59619766]
   [0.81901117 0.91792293 0.59824475]]

  [[0.61722773 0.00798019 0.67910435]
   [0.0899428  0.99873084 0.36475039]
   [0.55197416 0.25923369 0.59883985]]

  [[0.64473253 0.56582509 0.05307403]
   [0.85658534 0.97483502 0.62670432]
   [0.94130241 0.14511224 0.2623221 ]]]


 [[[0.11700951 0.27564612 0.96687832]
   [0.29951137 0.2632959  0.52804578]
   [0.60705034 0.98712846 0.89032515]]

  [[0.00273272 0.99889682 0.60025552]
   [0.42531668 0.14638614 0.30821971]
   [0.4559946  0.95861658 0.51063308]]

  [[0.12249758 0.98592893 0.11122947]
   [0.21745324 0.924399